## 1. A/B Testing

A/B testing membandingkan dua perlakuan atau dua versi berdasarkan ukuran hasil tertentu. Perbedaan yang diamati dapat ditulis sebagai

$$\Delta=\bar{x}_B-\bar{x}_A$$

Perbedaan ini perlu dibandingkan dengan variasi acak yang mungkin muncul jika kedua perlakuan sebenarnya tidak berbeda.

## 2. Hypothesis Tests

Null hypothesis (H₀) menyatakan tidak ada perbedaan/effect yang diuji. Alternative hypothesis (H₁) menyatakan terdapat perbedaan/effect.

Contoh:
$$H_0:\mu_A=\mu_B$$
$$H_1:\mu_A<\mu_B$$

Test statistic digunakan untuk mengukur besarnya effect relatif terhadap variasi acak.

## 3. Resampling dan Permutation Test

Resampling mengambil sampel berulang dari data untuk mempelajari variasi statistik. Bootstrap terutama digunakan untuk menilai reliability estimate, sedangkan permutation test digunakan untuk hypothesis testing.

Langkah permutation test: gabungkan data, acak, bagi kembali sesuai ukuran kelompok, hitung statistik, ulangi banyak kali, lalu bandingkan statistik observasi dengan distribusi hasil pengacakan.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

## 4. Example: Web Stickiness

Contoh menggunakan `session_times`

In [ ]:
session_times = pd.read_csv('session_times.csv')
session_times.head()

In [ ]:
ax = session_times.boxplot(by='Page', column='Time')
ax.set_xlabel('')
ax.set_ylabel('Time (in seconds)')
plt.suptitle('')
plt.show()

In [ ]:
mean_a = session_times[session_times.Page == 'Page A'].Time.mean()
mean_b = session_times[session_times.Page == 'Page B'].Time.mean()
mean_b - mean_a

Pertanyaannya adalah apakah selisih masih berada dalam range random variation.

In [ ]:
def perm_fun(x, nA, nB):
    n = nA + nB
    idx_B = set(random.sample(range(n), nB))
    idx_A = set(range(n)) - idx_B
    return x.loc[idx_B].mean() - x.loc[idx_A].mean()

In [ ]:
nA = 21
nB = 15
perm_diffs = [perm_fun(session_times.Time, nA, nB) for _ in range(1000)]
fig, ax = plt.subplots(figsize=(5, 5))
ax.hist(perm_diffs, bins=11, rwidth=0.9)
ax.axvline(x=mean_b - mean_a, color='black', lw=2)
ax.set_xlabel('Session time differences (in seconds)')
ax.set_ylabel('Frequency')
plt.show()

In [ ]:
np.mean(perm_diffs > mean_b - mean_a)

## 5. Statistical Significance, p-Value, Alpha, dan Error

Statistical significance menilai apakah hasil cukup ekstrem dibandingkan random variation di bawah H₀.

$$p=P(\text{hasil setidaknya se-ekstrem observasi}\mid H_0)$$

Alpha (α) adalah threshold yang ditetapkan sebelum pengujian, misalnya 0.05.

- Type I error: menyimpulkan ada effect padahal effect tidak ada.
- Type II error: gagal mendeteksi effect yang sebenarnya ada.

## 6. t-Test

t-test membandingkan mean dengan mempertimbangkan variasi dan ukuran sampel. Welch two-sample t-test tidak mengasumsikan varians kedua kelompok sama. Secara konseptual:

$$t=\frac{\bar{x}_A-\bar{x}_B}{\sqrt{s_A^2/n_A+s_B^2/n_B}}$$



In [ ]:
res = stats.ttest_ind(
    session_times[session_times.Page == 'Page A'].Time,
    session_times[session_times.Page == 'Page B'].Time,
    equal_var=False
)
print(f'p-value for single sided test: {res.pvalue / 2:.4f}')

## 7. Multiple Testing

Semakin banyak hypothesis test dilakukan, semakin besar peluang memperoleh false positive. Dengan 20 pengujian independen pada α=0.05:

$$P(\text{semua tidak signifikan})=0.95^{20}\approx0.36$$
$$P(\text{setidaknya satu false positive})=1-0.95^{20}\approx0.64$$

Fenomena ini disebut alpha inflation. Penyesuaian multiple comparisons dapat digunakan sesuai desain analisis.

## 8. Degrees of Freedom

Degrees of freedom adalah jumlah nilai yang masih bebas bervariasi setelah constraint diterapkan. Untuk variance sampel:

$$s^2=\frac{\sum(x_i-\bar{x})^2}{n-1}$$

sehingga df = n−1. df juga menjadi parameter pada distribusi t dan F.

## 9. ANOVA dan F-Statistic

ANOVA menguji kesamaan mean beberapa kelompok.

$$H_0:\mu_1=\mu_2=\cdots=\mu_k$$

Statistik F:
$$F=\frac{MS_{treatment}}{MS_{error}}$$

F membandingkan variasi antar-group dengan variasi residual.

In [ ]:
four_sessions = pd.read_csv('four_sessions.csv')
four_sessions.head()

In [ ]:
observed_variance = four_sessions.groupby('Page').mean(numeric_only=True).var()[0]
print('Observed means:', four_sessions.groupby('Page').mean(numeric_only=True).values.ravel())
print('Variance:', observed_variance)

def perm_test(df):
    df = df.copy()
    df['Time'] = np.random.permutation(df['Time'].values)
    return df.groupby('Page').mean(numeric_only=True).var()[0]

perm_variance = [perm_test(four_sessions) for _ in range(3000)]
print('Pr(Prob)', np.mean([var > observed_variance for var in perm_variance]))

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
model = smf.ols('Time ~ Page', data=four_sessions).fit()
aov_table = sm.stats.anova_lm(model)
aov_table

Pada ANOVA, `Df` adalah degrees of freedom, `Sum Sq` adalah sum of squares, `Mean Sq` adalah mean square, `F` adalah F-statistic, dan `PR(>F)` adalah p-value.

## 10. Two-Way ANOVA

Two-way ANOVA melibatkan dua faktor dan dapat memeriksa main effects serta interaction effect.

## 11. Chi-Square Test

Untuk data kategorikal, Pearson chi-square menggunakan

$$\chi^2=\sum\frac{(O-E)^2}{E}$$

Dengan `r` baris dan `c` kolom:
$$df=(r-1)(c-1)$$

Nilai chi-square yang besar menunjukkan observed counts semakin jauh dari expected counts.

In [ ]:
table_2x2 = np.array([[8, 2], [1, 9]])
oddsratio, pvalue = stats.fisher_exact(table_2x2)
print('Odds ratio:', oddsratio)
print('p-value:', pvalue)

## 12. Chi-Square dengan Resampling

Pendekatan resampling membangun distribusi null dengan mengacak data berkali-kali dan menghitung statistik chi-square pada setiap pengulangan. Proporsi statistik resampling yang lebih ekstrem daripada observasi digunakan sebagai p-value.

In [ ]:
def chi2(observed, expected):
    pearson_residuals = []
    for row, expect in zip(observed, expected):
        pearson_residuals.append([(observe - expect) ** 2 / expect for observe in row])
    return np.sum(pearson_residuals)

box = [1] * 34
box.extend([0] * 2966)
random.shuffle(box)
expected = [34 / 3, 1000 - 34 / 3]

def perm_fun_chi2(box):
    sample_clicks = [sum(random.sample(box, 1000)), sum(random.sample(box, 1000)), sum(random.sample(box, 1000))]
    sample_noclicks = [1000 - n for n in sample_clicks]
    return chi2([sample_clicks, sample_noclicks], expected)

click_counts = [12, 10, 12]
no_click_counts = [1000 - n for n in click_counts]
chi2observed = chi2([click_counts, no_click_counts], expected)
perm_chi2 = [perm_fun_chi2(box) for _ in range(2000)]
resampled_p_value = sum(perm_chi2 > chi2observed) / len(perm_chi2)
print(f'Observed chi2: {chi2observed:.4f}')
print(f'Resampled p-value: {resampled_p_value:.4f}')

## 13. Fisher's Exact Test

Fisher's exact test digunakan untuk contingency table, terutama ketika sampel kecil. Untuk tabel 2×2, pengujian mempertahankan marginal totals dan menghitung probabilitas konfigurasi tabel.

## 14. Multi-Arm Bandit Algorithm

Multi-arm bandit menangani beberapa pilihan yang reward-nya belum diketahui. Ada trade-off antara **exploration** (mencoba pilihan untuk memperoleh informasi) dan **exploitation** (menggunakan pilihan yang saat ini diperkirakan menghasilkan reward lebih tinggi).

## 15. Power and Sample Size

Effect size adalah ukuran effect minimum yang ingin dideteksi. Power adalah probabilitas mendeteksi effect tersebut dengan ukuran sampel dan karakteristik tertentu.

$$Power=P(\text{reject }H_0\mid H_1\text{ benar})$$

Power dipengaruhi effect size, sample size, variability, dan alpha. Effect kecil umumnya membutuhkan sampel lebih besar.